# Gather data

In [ ]:
import pandas as pd

import requests
import pandas as pd
from pathlib import Path
import time

import os

# Read from the environment -- never hardcode.
# Free key: https://collegefootballdata.com/key   then: export CFBD_API_KEY=...
API_KEY = os.environ["CFBD_API_KEY"]

BASE_URL = "https://api.collegefootballdata.com"

headers = {
    "Authorization": f"Bearer {API_KEY}"
}

DATA_DIR = Path("cfbd_data")
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
def get_recruits(year):
    url = f"{BASE_URL}/recruiting/players"

    params = {
        "year": year,
        "classification": "HighSchool"
    }

    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()

    df = pd.DataFrame(response.json())
    df["recruit_year"] = year

    return df


def get_player_stats(year, category):
    url = f"{BASE_URL}/stats/player/season"

    params = {
        "year": year,
        "category": category
    }

    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()

    df = pd.DataFrame(response.json())
    df["season"] = year
    df["category"] = category

    return df

In [ ]:
def cached_get_recruits(year):
    path = DATA_DIR / f"recruits_{year}.csv"

    if path.exists():
        print(f"Using cached recruits {year}")
        return pd.read_csv(path)

    print(f"Downloading recruits {year}")
    df = get_recruits(year)
    df.to_csv(path, index=False)
    time.sleep(0.25)
    return df


def cached_get_player_stats(year, category):
    path = DATA_DIR / f"player_stats_{category}_{year}.csv"

    if path.exists():
        print(f"Using cached {category} stats {year}")
        return pd.read_csv(path)

    print(f"Downloading {category} stats {year}")
    df = get_player_stats(year, category)
    df.to_csv(path, index=False)
    time.sleep(0.25)
    return df

In [ ]:
recruits_df = pd.concat(
    [cached_get_recruits(year) for year in range(2018, 2025)],
    ignore_index=True
)

stats_df = pd.concat(
    [
        cached_get_player_stats(year, category)
        for year in range(2018, 2026)
        for category in ["receiving"]
    ],
    ignore_index=True
)

# Cleaning and Merging

In [ ]:
# -----------------------
# 1. Clean names + positions
# -----------------------

def clean_name(s):
    return (
        s.astype(str)
        .str.lower()
        .str.strip()
        .str.replace(r"[^a-z\s]", "", regex=True)
        .str.replace(r"\s+", " ", regex=True)
    )

stats_df["name_clean"] = clean_name(stats_df["player"])
recruits_df["name_clean"] = clean_name(recruits_df["name"])

stats_df["position_clean"] = stats_df["position"].astype(str).str.upper().str.strip()
recruits_df["position_clean"] = recruits_df["position"].astype(str).str.upper().str.strip()

# -----------------------
# 2. Merge on name + position
# -----------------------

merged_name_pos = stats_df.merge(
    recruits_df,
    on=["name_clean", "position_clean"],
    how="left",
    suffixes=("_stat", "_recruit")
)

# -----------------------
# 3. Create eligibility year
# -----------------------

merged_name_pos["eligibility_year"] = (
    merged_name_pos["season"] - merged_name_pos["recruit_year"] + 1
)

# -----------------------
# 4. Keep only realistic eligibility years
# -----------------------

merged_name_pos = merged_name_pos[
    merged_name_pos["eligibility_year"].between(1, 5)
].copy()

# -----------------------
# 5. Prefer original school match when available
# -----------------------

merged_name_pos["team_clean"] = (
    merged_name_pos["team"]
    .astype(str)
    .str.lower()
    .str.strip()
)

merged_name_pos["committed_clean"] = (
    merged_name_pos["committedTo"]
    .astype(str)
    .str.lower()
    .str.strip()
)

merged_name_pos["school_match"] = (
    merged_name_pos["team_clean"] == merged_name_pos["committed_clean"]
).astype(int)

# -----------------------
# 6. Deduplicate player-seasons
# -----------------------
# Priority:
# 1. same school as original commitment
# 2. most recent plausible recruiting year
# 3. highest recruiting rating

merged_name_pos = merged_name_pos.sort_values(
    by=[
        "player",
        "season",
        "team",
        "school_match",
        "recruit_year",
        "rating"
    ],
    ascending=[
        True,
        True,
        True,
        False,
        False,
        False
    ]
)

merged_clean = merged_name_pos.drop_duplicates(
    subset=["player", "season", "team"],
    keep="first"
).copy()

# -----------------------
# 7. Check match quality
# -----------------------

print("Original stats rows:", len(stats_df))
print("Merged rows after cleaning:", len(merged_clean))
print("Matched rating rate:", merged_clean["rating"].notna().mean())

print("\nDuplicate counts after cleaning:")
print(
    merged_clean.groupby(["player", "season", "team"])
    .size()
    .value_counts()
    .sort_index()
)

merged_clean.head()

Original stats rows: 134245
Merged rows after cleaning: 7097
Matched rating rate: 1.0

Duplicate counts after cleaning:
1    7097
Name: count, dtype: int64


,season,playerId,player,position_stat,team,conference,category,statType,stat,name_clean,...,rating,city,stateProvince,country,hometownInfo,recruit_year,eligibility_year,team_clean,committed_clean,school_match
102142,2024,4918111,A'Marion Peterson,RB,USC,Big Ten,receiving,YPR,6.5,amarion peterson,...,0.8967,Wichita Falls,TX,USA,"{'latitude': 33.9137085, 'longitude': -98.4933...",2023.0,2.0,usc,usc,1
115884,2025,4918111,A'Marion Peterson,RB,UTSA,American Athletic,receiving,TD,0,amarion peterson,...,0.8967,Wichita Falls,TX,USA,"{'latitude': 33.9137085, 'longitude': -98.4933...",2023.0,3.0,utsa,usc,0
33365,2020,4372579,A.J. Abbott,WR,Wisconsin,Big Ten,receiving,LONG,9,aj abbott,...,0.8581,West Bloomfield,MI,USA,"{'latitude': 42.5679, 'longitude': -83.3733, '...",2018.0,3.0,wisconsin,wisconsin,1
38828,2021,4372579,A.J. Abbott,WR,Wisconsin,Big Ten,receiving,YPR,7.0,aj abbott,...,0.8581,West Bloomfield,MI,USA,"{'latitude': 42.5679, 'longitude': -83.3733, '...",2018.0,4.0,wisconsin,wisconsin,1
57830,2022,4372579,A.J. Abbott,WR,Western Michigan,Mid-American,receiving,REC,5,aj abbott,...,0.8581,West Bloomfield,MI,USA,"{'latitude': 42.5679, 'longitude': -83.3733, '...",2018.0,5.0,western michigan,wisconsin,0


## Checks to make sure matching succeeded

In [ ]:
print("Rows after name+position merge:", len(merged_name_pos))
print("Rows with rating:", merged_name_pos["rating"].notna().sum())
print("Rows with valid eligibility:", merged_name_pos["eligibility_year"].between(1, 5).sum())

Rows after name+position merge: 35725
Rows with rating: 35725
Rows with valid eligibility: 35725


In [ ]:
stats_names = set(stats_df["name_clean"])
recruit_names = set(recruits_df["name_clean"])

print("Unique stat names:", len(stats_names))
print("Unique recruit names:", len(recruit_names))
print("Name overlap:", len(stats_names & recruit_names))
print("Name overlap rate:", len(stats_names & recruit_names) / len(stats_names))

Unique stat names: 12702
Unique recruit names: 20921
Name overlap: 4196
Name overlap rate: 0.33034167847583057


# Create full pipeline

In [ ]:
import pandas as pd
import numpy as np

# -----------------------
# 1. Pivot stats_df from long to wide
# -----------------------

stats_wide = stats_df.pivot_table(
    index=[
        "player",
        "team",
        "conference",
        "season",
        "category",
        "playerId",
        "position"
    ],
    columns="statType",
    values="stat",
    aggfunc="first"
).reset_index()

stats_wide.columns.name = None

# -----------------------
# 2. Clean names + positions
# -----------------------

def clean_name(s):
    return (
        s.astype(str)
        .str.lower()
        .str.strip()
        .str.replace(r"[^a-z\s]", "", regex=True)
        .str.replace(r"\s+", " ", regex=True)
    )

stats_wide["name_clean"] = clean_name(stats_wide["player"])
recruits_df["name_clean"] = clean_name(recruits_df["name"])

stats_wide["position_clean"] = (
    stats_wide["position"].astype(str).str.upper().str.strip()
)

recruits_df["position_clean"] = (
    recruits_df["position"].astype(str).str.upper().str.strip()
)

# -----------------------
# 3. Merge stats with recruits on name + position
# -----------------------

merged_name_pos = stats_wide.merge(
    recruits_df,
    on=["name_clean", "position_clean"],
    how="left",
    suffixes=("_stat", "_recruit")
)

# -----------------------
# 4. Create eligibility year
# -----------------------

merged_name_pos["eligibility_year"] = (
    merged_name_pos["season"] - merged_name_pos["recruit_year"] + 1
)

# -----------------------
# 5. Keep only realistic eligibility years
# -----------------------

merged_name_pos = merged_name_pos[
    merged_name_pos["eligibility_year"].between(1, 5)
].copy()

# -----------------------
# 6. Prefer school match when available
# -----------------------

merged_name_pos["team_clean"] = (
    merged_name_pos["team"]
    .astype(str)
    .str.lower()
    .str.strip()
)

merged_name_pos["committed_clean"] = (
    merged_name_pos["committedTo"]
    .astype(str)
    .str.lower()
    .str.strip()
)

merged_name_pos["school_match"] = (
    merged_name_pos["team_clean"] == merged_name_pos["committed_clean"]
).astype(int)

# -----------------------
# 7. Deduplicate player-seasons
# -----------------------
# Priority:
# 1. same school as original commitment
# 2. most recent plausible recruiting year
# 3. highest recruiting rating

merged_name_pos = merged_name_pos.sort_values(
    by=[
        "player",
        "season",
        "team",
        "school_match",
        "recruit_year",
        "rating"
    ],
    ascending=[
        True,
        True,
        True,
        False,
        False,
        False
    ]
)

merged_clean = merged_name_pos.drop_duplicates(
    subset=["player", "season", "team"],
    keep="first"
).copy()

# -----------------------
# 8. Keep WR/TE only
# -----------------------

df = merged_clean[
    merged_clean["position_clean"].isin(["WR", "TE"])
].copy()

# -----------------------
# 9. Convert receiving columns to numeric
# -----------------------

for col in ["REC", "YDS", "TD"]:
    if col not in df.columns:
        df[col] = 0

    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Optional usage column, if you have it merged separately
if "Usage Overall" in df.columns:
    df["Usage Overall"] = pd.to_numeric(df["Usage Overall"], errors="coerce").fillna(0)
else:
    df["Usage Overall"] = 0

# -----------------------
# 10. Create yearly receiving score
# -----------------------

df["receiving_score"] = (
    df["YDS"]
    + 20 * df["TD"]
    + 5 * df["REC"]
)

# -----------------------
# 11. Create within-year percentiles
# -----------------------

df["usage_pct"] = (
    df.groupby("eligibility_year")["Usage Overall"]
    .rank(pct=True)
)

df["production_pct"] = (
    df.groupby("eligibility_year")["receiving_score"]
    .rank(pct=True)
)

# -----------------------
# 12. Create yearly impact score
# -----------------------

df["impact_score"] = (
    0.4 * df["usage_pct"]
    + 0.6 * df["production_pct"]
)

df["impact_percentile"] = (
    df.groupby("eligibility_year")["impact_score"]
    .rank(pct=True)
)

# -----------------------
# 13. Define yearly outcomes
# -----------------------

def classify_outcome(p):
    if p < 0.25:
        return "Bust"
    elif p < 0.60:
        return "Depth / Rotation"
    elif p < 0.85:
        return "Starter"
    else:
        return "Impact Player"

df["outcome"] = df["impact_percentile"].apply(classify_outcome)

# -----------------------
# 14. Check yearly distributions
# -----------------------

yearly_distribution = (
    df.groupby(["eligibility_year", "outcome"])
    .size()
    .unstack(fill_value=0)
)

print(yearly_distribution)

print("Final WR/TE player-season rows:", len(df))

df[
    [
        "player",
        "team",
        "season",
        "eligibility_year",
        "position_clean",
        "REC",
        "YDS",
        "TD",
        "Usage Overall",
        "receiving_score",
        "impact_percentile",
        "outcome",
        "stars",
        "rating",
        "ranking",
        "committedTo"
    ]
].head()

outcome           Bust  Depth / Rotation  Impact Player  Starter
eligibility_year                                                
1.0                219               300            131      219
2.0                314               438            188      311
3.0                301               424            181      304
4.0                249               352            151      251
5.0                184               258            111      184
Final WR/TE player-season rows: 5070


,player,team,season,eligibility_year,position_clean,REC,YDS,TD,Usage Overall,receiving_score,impact_percentile,outcome,stars,rating,ranking,committedTo
12,A.J. Abbott,Wisconsin,2020,3.0,WR,2,12,0,0,22,0.103719,Bust,3.0,0.8581,863.0,Wisconsin
13,A.J. Abbott,Wisconsin,2021,4.0,WR,1,7,0,0,12,0.042871,Bust,3.0,0.8581,863.0,Wisconsin
11,A.J. Abbott,Western Michigan,2022,5.0,WR,5,61,1,0,106,0.251696,Depth / Rotation,3.0,0.8581,863.0,Wisconsin
15,A.J. Barner,Indiana,2021,2.0,TE,14,162,1,0,252,0.635891,Starter,3.0,0.8522,1131.0,None
16,A.J. Barner,Indiana,2022,3.0,TE,27,193,3,0,388,0.647521,Starter,3.0,0.8522,1131.0,None


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestNeighbors

# -----------------------
# 1. Build recruit-level similarity table
# -----------------------

profile_cols = [
    "player",
    "name_clean",
    "stars",
    "rating",
    "ranking",
    "height",
    "weight",
    "position_clean",
    "committedTo",
    "recruit_year"
]

profiles = (
    df[profile_cols]
    .drop_duplicates(subset=["name_clean", "position_clean", "recruit_year"])
    .copy()
)

# -----------------------
# 2. Clean numeric columns
# -----------------------

numeric_features = [
    "stars",
    "rating",
    "ranking",
    "height",
    "weight"
]

categorical_features = [
    "position_clean",
    "committedTo"
]

for col in numeric_features:
    profiles[col] = pd.to_numeric(profiles[col], errors="coerce")

# -----------------------
# 3. Fit similarity model
# -----------------------

X_profiles = profiles[numeric_features + categorical_features].copy()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

X_processed = preprocessor.fit_transform(X_profiles)

knn = NearestNeighbors(
    n_neighbors=50,
    metric="euclidean"
)

knn.fit(X_processed)

# -----------------------
# 4. Year-by-year distribution function
# -----------------------

def yearly_player_outcome_distribution(
    stars,
    rating,
    ranking,
    height,
    weight,
    position,
    committed_to,
    n_neighbors=25,
    exclude_player=None,
):
    player_profile = pd.DataFrame([{
        "stars": stars,
        "rating": rating,
        "ranking": ranking,
        "height": height,
        "weight": weight,
        "position_clean": position,
        "committedTo": committed_to,
    }])

    player_processed = preprocessor.transform(player_profile)

    distances, indices = knn.kneighbors(
        player_processed,
        n_neighbors=n_neighbors
    )

    similar_profiles = profiles.iloc[indices[0]].copy()
    # Remove exact player match if using existing player mode
    if "player_name" in locals():
        similar_profiles = similar_profiles[
            similar_profiles["player"] != player_name
        ].copy()

    # Re-rank after removing self
    similar_profiles = similar_profiles.head(n_neighbors)
    similar_profiles["distance"] = distances[0]

    similar_keys = similar_profiles[
        ["name_clean", "position_clean", "recruit_year"]
    ]

    similar_years = df.merge(
        similar_keys,
        on=["name_clean", "position_clean", "recruit_year"],
        how="inner"
    )

    # -----------------------
# Safer distribution block
# -----------------------

    distribution = (
        similar_years
        .groupby(["eligibility_year", "outcome"])
        .size()
        .rename("count")
        .reset_index()
    )

    distribution["probability"] = (
        distribution["count"] /
        distribution.groupby("eligibility_year")["count"].transform("sum")
    )

    distribution_wide = (
        distribution
        .pivot_table(
            index="eligibility_year",
            columns="outcome",
            values="probability",
            fill_value=0
        )
        .reset_index()
    )

    for col in ["Bust", "Depth / Rotation", "Starter", "Impact Player"]:
        if col not in distribution_wide.columns:
            distribution_wide[col] = 0

    distribution_wide = distribution_wide[
        ["eligibility_year", "Bust", "Depth / Rotation", "Starter", "Impact Player"]
    ]

    distribution_wide = (
        distribution
        .pivot(index="eligibility_year", columns="outcome", values="probability")
        .fillna(0)
        .reset_index()
    )

    for col in ["Bust", "Depth / Rotation", "Starter", "Impact Player"]:
        if col not in distribution_wide.columns:
            distribution_wide[col] = 0

    distribution_wide = distribution_wide[
        ["eligibility_year", "Bust", "Depth / Rotation", "Starter", "Impact Player"]
    ]

    return similar_profiles, similar_years, distribution_wide

In [ ]:
df["eligibility_year"].value_counts().sort_index()

,count
eligibility_year,
1.0,869
2.0,1251
3.0,1210
4.0,1003
5.0,737


In [ ]:
similar_profiles, similar_years, distribution_wide = yearly_player_outcome_distribution(
    stars=4,
    rating=0.94,
    ranking=120,
    height=74,
    weight=190,
    position="WR",
    committed_to="Ohio State",
    n_neighbors=25
)

similar_years["eligibility_year"].value_counts().sort_index()

,count
eligibility_year,
1.0,18
2.0,20
3.0,18
4.0,12
5.0,8


In [ ]:
distribution_wide.round(2)

outcome,eligibility_year,Bust,Depth / Rotation,Starter,Impact Player
0,1.0,0.22,0.33,0.44,0.00
1,2.0,0.15,0.25,0.35,0.25
2,3.0,0.22,0.11,0.33,0.33
3,4.0,0.17,0.42,0.17,0.25
4,5.0,0.25,0.38,0.12,0.25


##Streamlit

In [ ]:
df.to_csv("yearly_wr_te_outcomes.csv", index=False)

# **Jump here** - datatable has been created already  

In [ ]:
  %%writefile app.py
  import streamlit as st
  import pandas as pd
  import numpy as np
  import matplotlib.pyplot as plt
  import json

  from sklearn.compose import ColumnTransformer
  from sklearn.pipeline import Pipeline
  from sklearn.preprocessing import OneHotEncoder, StandardScaler
  from sklearn.impute import SimpleImputer
  from sklearn.neighbors import NearestNeighbors

  st.set_page_config(page_title="Recruit Outcome Comps", layout="wide")

  OUTCOME_COLS = ["Bust", "Depth / Rotation", "Starter", "Impact Player"]

  @st.cache_resource
  def build_model():
      df = pd.read_csv("yearly_wr_te_outcomes.csv")
      numeric_features = ["stars", "rating", "ranking", "height", "weight"]
      categorical_features = ["position_clean", "committedTo"]

      profiles = (
          df[
              [
                  "player",
                  "name_clean",
                  "stars",
                  "rating",
                  "ranking",
                  "height",
                  "weight",
                  "position_clean",
                  "committedTo",
                  "recruit_year",
              ]
          ]
          .drop_duplicates(subset=["name_clean", "position_clean", "recruit_year"])
          .copy()
      )

      for col in numeric_features:
          profiles[col] = pd.to_numeric(profiles[col], errors="coerce")

      X = profiles[numeric_features + categorical_features].copy()

      preprocessor = ColumnTransformer(
          transformers=[
              (
                  "num",
                  Pipeline(
                      [
                          ("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler()),
                      ]
                  ),
                  numeric_features,
              ),
              (
                  "cat",
                  Pipeline(
                      [
                          ("imputer", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore")),
                      ]
                  ),
                  categorical_features,
              ),
          ]
      )

      X_processed = preprocessor.fit_transform(X)

      knn = NearestNeighbors(n_neighbors=50, metric="euclidean")
      knn.fit(X_processed)

      return df, profiles, preprocessor, knn, numeric_features, categorical_features


  df, profiles, preprocessor, knn, numeric_features, categorical_features = build_model()

  import json

  @st.cache_data
  def load_value_mapping():
      with open("value_mapping.json", "r") as f:
          return json.load(f)


  def compute_expected_value(distribution_wide, position, offer_amount):
      value_mapping = load_value_mapping()

      if position not in value_mapping:
          st.warning(f"No value mapping found for {position}. Using WR values.")
          position_values = value_mapping["WR"]
      else:
          position_values = value_mapping[position]

      ev_df = distribution_wide.copy()

      ev_df["expected_value"] = (
          ev_df["Bust"] * position_values["Bust"]
          + ev_df["Depth / Rotation"] * position_values["Depth / Rotation"]
          + ev_df["Starter"] * position_values["Starter"]
          + ev_df["Impact Player"] * position_values["Impact Player"]
      )

      ev_df["offer_amount"] = offer_amount
      ev_df["expected_surplus"] = ev_df["expected_value"] - offer_amount
      ev_df["roi"] = ev_df["expected_surplus"] / offer_amount

      return ev_df


  def yearly_player_outcome_distribution(
      stars,
      rating,
      ranking,
      height,
      weight,
      position,
      committed_to,
      n_neighbors=25,
      exclude_player = None
  ):
      player_profile = pd.DataFrame(
          [
              {
                  "stars": stars,
                  "rating": rating,
                  "ranking": ranking,
                  "height": height,
                  "weight": weight,
                  "position_clean": position,
                  "committedTo": committed_to,
              }
          ]
      )

      player_processed = preprocessor.transform(player_profile)

      distances, indices = knn.kneighbors(
          player_processed,
          n_neighbors=n_neighbors,
      )

      similar_profiles = profiles.iloc[indices[0]].copy()
      similar_profiles["distance"] = distances[0]

      if exclude_player is not None:
          similar_profiles = similar_profiles[
              similar_profiles["player"] != exclude_player
          ].copy()

      similar_profiles = similar_profiles.head(n_neighbors)

      similar_keys = similar_profiles[
          ["name_clean", "position_clean", "recruit_year"]
      ]

      similar_years = df.merge(
          similar_keys,
          on=["name_clean", "position_clean", "recruit_year"],
          how="inner",
      )

      distribution = (
          similar_years.groupby(["eligibility_year", "outcome"])
          .size()
          .rename("count")
          .reset_index()
      )

      distribution["probability"] = (
          distribution["count"]
          / distribution.groupby("eligibility_year")["count"].transform("sum")
      )

      distribution_wide = (
          distribution.pivot_table(
              index="eligibility_year",
              columns="outcome",
              values="probability",
              fill_value=0,
          )
          .reset_index()
      )

      for col in OUTCOME_COLS:
          if col not in distribution_wide.columns:
              distribution_wide[col] = 0

      distribution_wide = (
          distribution_wide.set_index("eligibility_year")
          .reindex([1, 2, 3, 4, 5], fill_value=0)
          .reset_index()
      )

      distribution_wide = distribution_wide[
          ["eligibility_year"] + OUTCOME_COLS
      ]

      sample_sizes = (
          similar_years.groupby("eligibility_year")
          .size()
          .reindex([1, 2, 3, 4, 5], fill_value=0)
          .reset_index(name="sample_size")
      )

      return similar_profiles, similar_years, distribution_wide, sample_sizes


  def render_distribution_visuals(distribution_wide, sample_sizes):
      plot_df = distribution_wide.copy()

      for col in OUTCOME_COLS:
          if col not in plot_df.columns:
              plot_df[col] = 0

      row_sums = plot_df[OUTCOME_COLS].sum(axis=1)

      plot_df[OUTCOME_COLS] = (
          plot_df[OUTCOME_COLS]
          .div(row_sums.replace(0, pd.NA), axis=0)
          .fillna(0)
      )

      table_df = plot_df.merge(sample_sizes, on="eligibility_year", how="left")

      for col in OUTCOME_COLS:
          table_df[col] = (table_df[col] * 100).round(1).astype(str) + "%"

      table_df = table_df.rename(
          columns={
              "eligibility_year": "Year",
              "sample_size": "Sample Size",
          }
      )

      st.subheader("Year-by-Year Outcome Distribution")
      st.dataframe(table_df, use_container_width=True)

      fig, ax = plt.subplots(figsize=(10, 6))

      years = plot_df["eligibility_year"].astype(int)

      colors = {
          "Bust": "#8B0000",
          "Depth / Rotation": "#808080",
          "Starter": "#1f77b4",
          "Impact Player": "#2ca02c",
      }

      bottom = pd.Series([0] * len(plot_df), dtype=float)

      for outcome in OUTCOME_COLS:
          values = plot_df[outcome].astype(float)

          ax.bar(
              years,
              values,
              bottom=bottom,
              label=outcome,
              color=colors[outcome],
          )

          bottom = bottom + values

      ax.set_ylim(0, 1)
      ax.set_xticks([1, 2, 3, 4, 5])
      ax.set_xlabel("Eligibility Year")
      ax.set_ylabel("Probability")
      ax.set_title("Historical Outcome Distribution by Eligibility Year")
      ax.legend(title="Outcome", bbox_to_anchor=(1.05, 1), loc="upper left")
      ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

      st.pyplot(fig)

      plot_df["expected_value"] = (
          0 * plot_df["Bust"]
          + 1 * plot_df["Depth / Rotation"]
          + 2 * plot_df["Starter"]
          + 3 * plot_df["Impact Player"]
      )

      fig2, ax2 = plt.subplots(figsize=(9, 5))

      ax2.plot(
          years,
          plot_df["expected_value"],
          marker="o",
          linewidth=3,
      )

      ax2.set_ylim(0, 3)
      ax2.set_xticks([1, 2, 3, 4, 5])
      ax2.set_xlabel("Eligibility Year")
      ax2.set_ylabel("Expected Outcome Value")
      ax2.set_title("Expected Development Trajectory")
      ax2.set_yticks([0, 1, 2, 3])
      ax2.set_yticklabels(["Bust", "Depth", "Starter", "Impact"])

      st.pyplot(fig2)


  st.title("Recruit Outcome Distribution Tool")

  st.write(
      "Enter a WR/TE recruiting profile to find similar historical players "
      "and estimate year-by-year career outcome probabilities."
  )

  st.sidebar.header("Player Profile")

  input_mode = st.sidebar.radio(
      "Input Mode",
      ["Use Existing Player", "Manual Entry"],
      key="input_mode",
  )

  if input_mode == "Use Existing Player":
      player_options = sorted(profiles["player"].dropna().unique())

      selected_player = st.sidebar.selectbox(
          "Select Player",
          player_options,
          key="existing_player_select",
      )

      selected_row = profiles[profiles["player"] == selected_player].iloc[0]

      player_name = selected_row["player"]
      stars = selected_row["stars"]
      rating = selected_row["rating"]
      ranking = selected_row["ranking"]
      height = selected_row["height"]
      weight = selected_row["weight"]
      position = selected_row["position_clean"]
      committed_to = selected_row["committedTo"]

      st.sidebar.markdown("### Loaded Profile")
      st.sidebar.markdown(
          f"""
          **Stars:** {stars}
          **Rating:** {rating:.4f}
          **Ranking:** {ranking}
          **Height:** {height}
          **Weight:** {weight}
          **Position:** {position}
          **School:** {committed_to}
          """
      )

  else:
      player_name = st.sidebar.text_input(
          "Player Name",
          "Example Player",
          key="manual_player_name",
      )

      stars = st.sidebar.slider(
          "Stars",
          2,
          5,
          4,
          key="manual_stars",
      )

      rating = st.sidebar.number_input(
          "247 Composite Rating",
          min_value=0.0,
          max_value=1.0,
          value=0.940,
          step=0.001,
          format="%.3f",
          key="manual_rating",
      )

      ranking = st.sidebar.number_input(
          "National Ranking",
          min_value=1,
          max_value=5000,
          value=120,
          key="manual_ranking",
      )

      height = st.sidebar.number_input(
          "Height in Inches",
          min_value=60,
          max_value=90,
          value=74,
          key="manual_height",
      )

      weight = st.sidebar.number_input(
          "Weight",
          min_value=120,
          max_value=400,
          value=190,
          key="manual_weight",
      )

      position = st.sidebar.selectbox(
          "Position",
          sorted(profiles["position_clean"].dropna().unique()),
          key="manual_position",
      )

      committed_to = st.sidebar.selectbox(
          "Committed To",
          sorted(profiles["committedTo"].dropna().unique()),
          key="manual_school",
      )

  n_neighbors = st.sidebar.slider(
      "Comparable Players",
      5,
      50,
      25,
      key="neighbor_slider",
  )

  offer_amount_k = st.sidebar.slider(
    "Proposed Offer ($000s)",
    min_value=0,
    max_value=1000,
    value=250,
    step=25,
    key="offer_amount_k"
    )

  offer_amount = offer_amount_k * 1000

  st.sidebar.markdown(
      f"""
      <h2 style='margin-top: -10px;'>
          Offer: ${offer_amount:,.0f}
      </h2>
      """,
      unsafe_allow_html=True
  )

  run_button = st.sidebar.button(
      "Find Historical Comps",
      key="find_comps_button",
  )

if run_button:

    similar_profiles, similar_years, distribution_wide, sample_sizes = (
        yearly_player_outcome_distribution(
            stars=stars,
            rating=rating,
            ranking=ranking,
            height=height,
            weight=weight,
            position=position,
            committed_to=committed_to,
            n_neighbors=n_neighbors,
            exclude_player=player_name if input_mode == "Use Existing Player" else None
        )
    )

    st.subheader(f"Historical Comps for {player_name}")

    profile_col, metric_col = st.columns(2)

    with profile_col:
        st.markdown(
            f"""
            **Profile:** {stars}★ {position}

            **Rating:** {rating:.4f}

            **Ranking:** {ranking}

            **School:** {committed_to}
            """
        )

    with metric_col:
        st.metric("Comparable Players", n_neighbors)

    # -----------------------
    # Investment evaluation
    # -----------------------

    ev_df = compute_expected_value(
        distribution_wide=distribution_wide,
        position=position,
        offer_amount=offer_amount
    )

    display_ev = ev_df[
        [
            "eligibility_year",
            "expected_value",
            "offer_amount",
            "expected_surplus",
            "roi"
        ]
    ].copy()

    display_ev["expected_value"] = (
        display_ev["expected_value"]
        .map("${:,.0f}".format)
    )

    display_ev["offer_amount"] = (
        display_ev["offer_amount"]
        .map("${:,.0f}".format)
    )

    display_ev["expected_surplus"] = (
        display_ev["expected_surplus"]
        .map("${:,.0f}".format)
    )

    display_ev["roi"] = (
        (ev_df["roi"] * 100)
        .round(1)
        .astype(str) + "%"
    )

    display_ev = display_ev.rename(columns={
        "eligibility_year": "Year",
        "expected_value": "Expected Value",
        "offer_amount": "Offer",
        "expected_surplus": "Expected Surplus",
        "roi": "ROI"
    })

    st.subheader("Investment Evaluation")

    st.dataframe(display_ev, use_container_width=True)
    # -----------------------
    # Investment summary
    # -----------------------

    total_ev = ev_df["expected_value"].max()

    total_compensation = offer_amount

    net_value = total_ev - total_compensation

    roi_multiple = (
        total_ev / total_compensation
        if total_compensation > 0
        else np.nan
    )

    downside_probability = distribution_wide["Bust"].mean()

    st.subheader("Investment Summary")

    col1, col2, col3, col4 = st.columns(4)

    col1.metric(
        "Expected Value Ceiling",
        f"${total_ev:,.0f}"
    )

    col2.metric(
        "Net Value",
        f"${net_value:,.0f}"
    )

    col3.metric(
        "ROI",
        f"{roi_multiple:.2f}x"
    )

    col4.metric(
        "Downside Probability",
        f"{downside_probability:.1%}"
    )

    avg_expected_value = ev_df["expected_value"].mean()

    avg_surplus = avg_expected_value - offer_amount

    if avg_surplus > 0:

        st.success(
            f"""
            Estimated positive investment.

            Average expected value:
            ${avg_expected_value:,.0f}

            Estimated surplus above offer:
            ${avg_surplus:,.0f}
            """
        )

    else:

        st.error(
            f"""
            Estimated negative investment.

            Average expected value:
            ${avg_expected_value:,.0f}

            Estimated deficit below offer:
            ${abs(avg_surplus):,.0f}
            """
        )

    # -----------------------
    # Charts
    # -----------------------

    render_distribution_visuals(
        distribution_wide,
        sample_sizes
    )

    # -----------------------
    # Similar profiles
    # -----------------------

    st.subheader("Most Similar Historical Players")

    display_cols = [
        "player",
        "stars",
        "rating",
        "ranking",
        "height",
        "weight",
        "position_clean",
        "committedTo",
        "recruit_year",
        "distance",
    ]

    st.dataframe(
        similar_profiles[display_cols],
        use_container_width=True,
    )

    # -----------------------
    # Player-year rows
    # -----------------------

    st.subheader("Matched Player-Year Rows Used")

    year_display_cols = [
        "player",
        "team",
        "season",
        "eligibility_year",
        "outcome",
        "REC",
        "YDS",
        "TD",
        "receiving_score",
    ]

    available_year_cols = [
        col for col in year_display_cols if col in similar_years.columns
    ]

    st.dataframe(
        similar_years[available_year_cols],
        use_container_width=True,
    )

else:

    st.info(
        "Enter or load a player profile, then click Find Historical Comps."
    )

Overwriting app.py


In [ ]:
# import sys
# !{sys.executable} -m pip install pyngrok
# !pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 49.8 MB/s eta 0:00:00


In [ ]:
!pkill -f streamlit
!pkill -f ngrok

In [ ]:
!streamlit run app.py --server.port 8502 --server.address 0.0.0.0 > streamlit.log 2>&1 &

In [ ]:
!streamlit run app.py --server.port 8502 &
!curl http://localhost:8502



2026-05-22 00:06:41.385 Port 8502 is not available
<!--
 Copyright (c) Streamlit Inc. (2018-2022) Snowflake Inc. (2022-2026)

 Licensed under the Apache License, Version 2.0 (the "License");
 you may not use this file except in compliance with the License.
 You may obtain a copy of the License at

     http://www.apache.org/licenses/LICENSE-2.0

 Unless required by applicable law or agreed to in writing, software
 distributed under the License is distributed on an "AS IS" BASIS,
 WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
 See the License for the specific language governing permissions and
 limitations under the License.
-->

<!DOCTYPE html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />
    <meta
      name="viewport"
      content="width=device-width, initial-scale=1, shrink-to-fit=no"
    />
    <link rel="shortcut icon" href="./favicon.png" />
    <link
      rel="preload"
      href="./static/media/SourceSansVF-Upright.ttf.BsWL4Kly.woff2"
  

In [ ]:
from pyngrok import ngrok

ngrok.kill()
ngrok.set_auth_token("3DQIm2K2lAlED6WiL22Y9gPjAc8_c3wJAkvw3mryTRNxMhpj")
public_url = ngrok.connect(8502)
print(public_url)

NgrokTunnel: "https://doormat-unwanted-image.ngrok-free.dev" -> "http://localhost:8502"


In [ ]:
%%writefile value_mapping.json
{
  "WR": {
    "Bust": 0,
    "Depth / Rotation": 100000,
    "Starter": 400000,
    "Impact Player": 1200000
  },
  "TE": {
    "Bust": 0,
    "Depth / Rotation": 80000,
    "Starter": 300000,
    "Impact Player": 900000
  }
}

Writing value_mapping.json


## Read me

In [ ]:
%%writefile README.md
# Phase 1 Documentation

## Goal
For a given WR/TE recruiting profile, estimate the historical distribution of career outcomes across eligibility Years 1–5.

## Data Sources
- CollegeFootballData API
- Recruiting data: high school recruit profiles
- Player season stats: receiving statistics
- Optional usage data if available

## Pipeline
1. Pull recruiting data across multiple recruiting classes.
2. Pull player season stats across multiple seasons.
3. Pivot player stats from long to wide format.
4. Clean player names and positions.
5. Merge recruiting profiles to player-season stats using cleaned name + position.
6. Filter to realistic eligibility years, 1–5.
7. Label each player-season with an outcome category.
8. Save final dataset to `yearly_wr_te_outcomes.csv`.

## Model
The model uses nearest-neighbor similarity.

Input features:
- Stars
- 247 composite rating
- National ranking
- Height
- Weight
- Position
- Committed school

For an input profile, the system finds the most similar historical recruits and computes the empirical outcome distribution for each eligibility year.

## Output
For each eligibility year, the model returns probabilities across:
- Bust
- Depth / Rotation
- Starter
- Impact Player

## Known Limitations
- Current version only supports WR/TE.
- Matching is based on cleaned name + position, not perfect player IDs.
- Transfer portal movement is not fully handled.
- Current outcome labels are percentile-based and should later be replaced with fixed contribution thresholds.
- Players without recruiting profiles are excluded.
- Similarity estimates depend heavily on available historical matches.
- Year 5 estimates may have smaller sample sizes.

## Reproduction Instructions
1. Run the data-pull notebook.
2. Run the cleaning/merge/yearly outcome pipeline.
3. Save the final dataset:

```python
df.to_csv("yearly_wr_te_outcomes.csv", index=False)

Writing README.md
